# Rule vs transformer evaluation

Keep prototypes here, not in `src/automotive_ner`. Start from the rule baseline and labeled fixture. Optional transformer experiments require separately approved model packages/weights; do not download data or weights that contain private customer content.

Install dependencies

In [1]:
import sys

!{sys.executable} -m pip install -U spacy transformers datasets evaluate seqeval accelerate

     ---------------------------------------- 0.0/43.6 kB ? eta -:--:--
     -------------------------------------- 43.6/43.6 kB 709.9 kB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/57.4 kB ? eta -:--:--
     ----------------------------------- ---- 51.2/57.4 kB 2.6 MB/s eta 0:00:01
     ----------------------------------- ---- 51.2/57.4 kB 2.6 MB/s eta 0:00:01
     -------------------------------------- 57.4/57.4 kB 432.3 kB/s eta 0:00:00
     ---------------------------------------- 0.0/141.5 kB ? eta -:--:--
     ----------------------


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Environment verification

In [2]:
import sys
import pandas as pd
import numpy as np
import spacy
import torch
import transformers

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("spaCy:", spacy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Pandas: 3.1.1
spaCy: 3.8.18
PyTorch: 2.15.0+cpu
Transformers: 5.23.0
CUDA available: False


Load the annotations

In [6]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

ANNOTATION_PATH = DATA_DIR / "ner_annotations.jsonl"

assert ANNOTATION_PATH.exists(), (
    f"Annotation file missing: {ANNOTATION_PATH}. "
    "Complete the human annotation step first."
)

records = []

with open(ANNOTATION_PATH, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if line.strip():
            record = json.loads(line)
            records.append(record)

print("Total annotated documents:", len(records))
print(records[0])

Total annotated documents: 3
{'id': '10001', 'text': 'The transmission started slipping.', 'entities': [{'start': 4, 'end': 16, 'label': 'CAR_PART'}, {'start': 26, 'end': 34, 'label': 'SYMPTOM'}]}


Validate the annotation contract

In [7]:
VALID_LABELS = {"CAR_PART", "SYMPTOM"}

def validate_record(record):
    required = {"id", "text", "entities"}

    assert required.issubset(record.keys()), (
        f"Missing fields in record: {record.get('id')}"
    )

    text = record["text"]
    entities = record["entities"]

    assert isinstance(text, str) and text.strip()
    assert isinstance(entities, list)

    previous_end = -1

    for entity in sorted(entities, key=lambda x: x["start"]):
        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        assert isinstance(start, int) and isinstance(end, int)
        assert 0 <= start < end <= len(text)
        assert label in VALID_LABELS

        # Our initial NER schema doesn't allow nested/overlapping entities.
        assert start >= previous_end, (
            f"Overlapping annotations in {record['id']}"
        )

        # Detect accidental whitespace around annotated spans.
        assert text[start:end] == text[start:end].strip()

        previous_end = end

for record in records:
    validate_record(record)

ids = [record["id"] for record in records]

assert len(ids) == len(set(ids)), "Duplicate document IDs!"

print("All annotations passed validation.")

All annotations passed validation.


In [8]:
from collections import Counter

entity_counts = Counter(
    entity["label"]
    for record in records
    for entity in record["entities"]
)

print("Entity distribution:", entity_counts)

print(
    "Documents with at least one entity:",
    sum(bool(r["entities"]) for r in records)
)

print(
    "Documents without entities:",
    sum(not r["entities"] for r in records)
)

Entity distribution: Counter({'CAR_PART': 2, 'SYMPTOM': 2})
Documents with at least one entity: 2
Documents without entities: 1


Split the documents

In [9]:
import re
from sklearn.model_selection import GroupShuffleSplit

def normalize_for_grouping(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

groups = [
    normalize_for_grouping(record["text"])
    for record in records
]

indices = np.arange(len(records))

# Reserve 15% for final testing
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=42
)

train_val_idx, test_idx = next(
    split1.split(indices, groups=groups)
)

# Reserve approximately 15% of the complete dataset for validation
# 15 / 85 = 0.17647
train_val_groups = [groups[i] for i in train_val_idx]

split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.17647,
    random_state=42
)

train_relative, val_relative = next(
    split2.split(
        train_val_idx,
        groups=train_val_groups
    )
)

train_idx = train_val_idx[train_relative]
val_idx = train_val_idx[val_relative]

train_records = [records[i] for i in train_idx]
val_records = [records[i] for i in val_idx]
test_records = [records[i] for i in test_idx]

print("Train:", len(train_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

Train: 1
Validation: 1
Test: 1


Verify there's no data leakage

In [10]:
train_groups = {groups[i] for i in train_idx}
val_groups = {groups[i] for i in val_idx}
test_groups = {groups[i] for i in test_idx}

assert not (train_groups & val_groups)
assert not (train_groups & test_groups)
assert not (val_groups & test_groups)

print("No exact-text leakage across splits.")

No exact-text leakage across splits.


In [11]:
def split_summary(name, dataset):
    counts = Counter(
        entity["label"]
        for record in dataset
        for entity in record["entities"]
    )

    print(f"\n{name}")
    print("Documents:", len(dataset))
    print("Entities:", dict(counts))
    print(
        "Documents with entities:",
        sum(bool(r["entities"]) for r in dataset)
    )

split_summary("TRAIN", train_records)
split_summary("VALIDATION", val_records)
split_summary("TEST", test_records)


TRAIN
Documents: 1
Entities: {'CAR_PART': 1, 'SYMPTOM': 1}
Documents with entities: 1

VALIDATION
Documents: 1
Entities: {}
Documents with entities: 0

TEST
Documents: 1
Entities: {'CAR_PART': 1, 'SYMPTOM': 1}
Documents with entities: 1


Extract gold-standard spans

In [12]:
def gold_spans(record):
    return {
        (e["start"], e["end"], e["label"])
        for e in record["entities"]
    }

Implement exact entity-level metrics

In [13]:
def calculate_entity_metrics(gold_records, predictions):

    assert len(gold_records) == len(predictions)

    labels = sorted(VALID_LABELS)

    gold_all = set()
    pred_all = set()

    for doc_idx, (record, predicted_entities) in enumerate(
        zip(gold_records, predictions)
    ):

        for entity in gold_spans(record):
            gold_all.add((doc_idx, *entity))

        for entity in predicted_entities:
            pred_all.add((doc_idx, *entity))

    true_positives = gold_all & pred_all
    false_positives = pred_all - gold_all
    false_negatives = gold_all - pred_all

    def score(tp, fp, fn):
        precision = tp / (tp + fp) if tp + fp else 0
        recall = tp / (tp + fn) if tp + fn else 0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0
        )

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": tp + fn
        }

    results = {
        "overall": score(
            len(true_positives),
            len(false_positives),
            len(false_negatives)
        )
    }

    for label in labels:
        tp = sum(x[3] == label for x in true_positives)
        fp = sum(x[3] == label for x in false_positives)
        fn = sum(x[3] == label for x in false_negatives)

        results[label] = score(tp, fp, fn)

    return results

Pretty-print evaluation results

In [14]:
def display_metrics(metrics, model_name):

    print(f"\n{'='*55}")
    print(f"MODEL: {model_name}")
    print(f"{'='*55}")

    table = pd.DataFrame(metrics).T
    display(table.round(4))

    return table

Define our seed vocabulary

In [15]:
PART_ALIASES = [
    "brake pad",
    "brake pads",
    "brake pedal",
    "brake master cylinder",
    "brake booster",
    "brake rotor",
    "brake disc",

    "engine",
    "engine block",
    "engine mount",
    "engine oil",
    "cylinder head",
    "spark plug",
    "ignition coil",
    "alternator",
    "starter motor",
    "fuel pump",
    "water pump",
    "oil pump",
    "catalytic converter",

    "transmission",
    "gearbox",
    "clutch",
    "clutch pedal",
    "torque converter",

    "steering wheel",
    "power steering",
    "steering rack",
    "tie rod",
    "wheel bearing",
    "suspension",
    "shock absorber",
    "strut",

    "air conditioning",
    "a/c",
    "ac unit",
    "compressor",
    "radiator",
    "thermostat",

    "battery",
    "ecu",
    "electronic control unit",
    "abs module",
    "abs sensor",
    "infotainment system",
    "touchscreen",
    "backup camera",
    "rearview camera",
    "airbag",
    "seat belt",
    "tire",
    "tyre"
]

SYMPTOM_ALIASES = [
    "overheating",
    "overheated",
    "stalling",
    "stalled",
    "squeaking",
    "squealing",
    "leaking",
    "leakage",
    "slipping",
    "misfiring",
    "vibrating",
    "vibration",
    "shaking",
    "smoking",
    "failure",
    "failed",
    "not starting",
    "won't start",
    "hard starting",
    "loss of power",
    "power loss",
    "rough idling",
    "jerking",
    "grinding",
    "rattling",
    "burning smell",
    "warning light"
]

print("CAR_PART aliases:", len(PART_ALIASES))
print("SYMPTOM aliases:", len(SYMPTOM_ALIASES))

CAR_PART aliases: 52
SYMPTOM aliases: 27


Create the spaCy rule-based recognizer

In [16]:
import spacy

rule_nlp = spacy.blank("en")

ruler = rule_nlp.add_pipe(
    "entity_ruler",
    config={
        "phrase_matcher_attr": "LOWER",
        "overwrite_ents": True
    }
)

patterns = []

for term in PART_ALIASES:
    patterns.append({
        "label": "CAR_PART",
        "pattern": term
    })

for term in SYMPTOM_ALIASES:
    patterns.append({
        "label": "SYMPTOM",
        "pattern": term
    })

ruler.add_patterns(patterns)

print("Pipeline:", rule_nlp.pipe_names)
print("Patterns:", len(ruler.patterns))

Pipeline: ['entity_ruler']
Patterns: 79


Test the rule recognizer manually

In [17]:
test_sentence = (
    "The brake pads started squeaking and "
    "the transmission began slipping."
)

doc = rule_nlp(test_sentence)

for ent in doc.ents:
    print(
        f"Text: {ent.text:25} "
        f"Label: {ent.label_:10} "
        f"Start: {ent.start_char} "
        f"End: {ent.end_char}"
    )

Text: brake pads                Label: CAR_PART   Start: 4 End: 14
Text: squeaking                 Label: SYMPTOM    Start: 23 End: 32
Text: transmission              Label: CAR_PART   Start: 41 End: 53
Text: slipping                  Label: SYMPTOM    Start: 60 End: 68


In [18]:
ambiguous_examples = [
    "I cannot drive the vehicle anymore.",
    "The drive shaft is damaged.",
    "The engine is overheating.",
    "The battery warning light appeared.",
    "The car feels different."
]

for text in ambiguous_examples:
    doc = rule_nlp(text)

    print("\n", text)
    print([(ent.text, ent.label_) for ent in doc.ents])


 I cannot drive the vehicle anymore.
[]

 The drive shaft is damaged.
[]

 The engine is overheating.
[('engine', 'CAR_PART'), ('overheating', 'SYMPTOM')]

 The battery warning light appeared.
[('battery', 'CAR_PART'), ('warning light', 'SYMPTOM')]

 The car feels different.
[]


Build a reusable prediction function

In [19]:
def spacy_predict(nlp, texts, batch_size=32):

    predictions = []

    for doc in nlp.pipe(texts, batch_size=batch_size):

        entities = [
            (
                ent.start_char,
                ent.end_char,
                ent.label_
            )
            for ent in doc.ents
        ]

        predictions.append(entities)

    return predictions

Evaluate the rule baseline

In [20]:
val_texts = [r["text"] for r in val_records]

rule_val_predictions = spacy_predict(
    rule_nlp,
    val_texts
)

rule_val_metrics = calculate_entity_metrics(
    val_records,
    rule_val_predictions
)

display_metrics(
    rule_val_metrics,
    "spaCy EntityRuler"
)


MODEL: spaCy EntityRuler


,precision,recall,f1,support
overall,0.0,0.0,0.0,0.0
CAR_PART,0.0,0.0,0.0,0.0
SYMPTOM,0.0,0.0,0.0,0.0


,precision,recall,f1,support
overall,0.0,0.0,0.0,0.0
CAR_PART,0.0,0.0,0.0,0.0
SYMPTOM,0.0,0.0,0.0,0.0


Inspect false positives and false negatives

In [21]:
def inspect_errors(records, predictions):

    rows = []

    for idx, (record, predicted) in enumerate(
        zip(records, predictions)
    ):

        gold = gold_spans(record)
        pred = set(predicted)

        for start, end, label in pred - gold:
            rows.append({
                "id": record["id"],
                "type": "FALSE_POSITIVE",
                "predicted_text": record["text"][start:end],
                "predicted_label": label,
                "context": record["text"]
            })

        for start, end, label in gold - pred:
            rows.append({
                "id": record["id"],
                "type": "FALSE_NEGATIVE",
                "predicted_text": record["text"][start:end],
                "predicted_label": label,
                "context": record["text"]
            })

    return pd.DataFrame(rows)

rule_errors = inspect_errors(
    val_records,
    rule_val_predictions
)

display(rule_errors.head(30))

,id,type,predicted_text,predicted_label,context
0,10003,FALSE_POSITIVE,failed,SYMPTOM,The vehicle failed to start.
